<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/ML_Credit_Scoring_Model_for_Private_Middle_Market_Borrowers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Executive Summary

In private credit and commercial middle-market lending ($10M - $100M EBITDA), borrowers lack public credit ratings from S&P, Moody's, or Fitch. Front office credit analytics and risk engineering teams deploy Graident Boosted Decision Trees (e.g., XGBoost, LightGBM) to automate internal **Probability of Default (PD)** scoring by ingesting unrated financial ratios, cash flow dynamics, and qualtitative sponsor metrics.

While XGBoost significantly improves discriminative power (AUC / Gini) over classical logistic regression by capturing non-linear interactions (e.g., interest coverage degradation under rising leverage), Model Risk Management (MRM) validators must address severe stuctural vulnerabilities:

1. **Non-Monotonic Decision Boundaries**: Unconstrained decision trees can create non-monotonic PD predictions across continuous risk factors (e.g., a local region where higher leverage *decreases* predicted default risk), violating economic sanity and regulatory SR 11-7 conceptual soundness.
2. **Panel Data Leakage in Cross Validation**: Middle-market borrower datasets consist of panel time-series data. Standard random K-fold cross-validation causes severe data leakage across time horizons, causing front-office teams to drastically overstate out-of-sample performance.
3. **Probability Calibration Drift in Low Default Frequency (LDF) Portfolios**: Unrated middle-market portfolios often exhibit low default counts. Raw tree probabilities cluster near zero or one, requiring post-hoc probability calibration (Isotonic Regression / Platt Scaling) to match historical portfolio long-run average (LRA) default rates.
4. **XAI Instability & FCRA Adverse Action Compliance**: Generating stable, legally defensible adverse action reason codes under FCRA/ECOA using local feature attribution methods (SHAP).
---
# 2. Mathematical Framework

### 1. XGBoost Objective with Enforced Monotonicity Constraints
For an unrated obligor dataset $\mathbb{D} = {(x_{i}, y_{i})}$, where $y_{i} \in \{0, 1\}$ denotes default within a 1-year horizon, the objective function at tree step $t$ is:

$$\mathcal{L}^{(t)} = \sum^{n}_{i=1} l\left(y_{i}, \hat{y}^{(t-1)}_{i} + f_{t}(\mathbf{x}_{i})\right) + \gamma T + \frac{1}{2}\lambda \sum^{T}_{j=1}w^{2}_{j}$$

To ensure economic monotonicity, a constraint vector $\mathbf{m} \in \{-1, 0, +1 \}^{p}$ is imposed on feature dimension $j \in \{1,...,p\}$ during tree split searching:

$$x_{i,j} \ge x_{k, j} \Rightarrow f(\mathbf{x}_{i}) \le f(\mathbf{x}_{k}), \qquad \text{if  } m_{j}=-1 \text{  (e.g., Interest Coverage Ratio)}$$

$$x_{i,j} \ge x_{k, j} \Rightarrow f(\mathbf{x}_{i}) \ge f(\mathbf{x}_{k}), \qquad \text{if  } m_{j}=+1 \text{  (e.g., Leverage Ratio Total Debt)}$$

### 2. Monotonicity Violation Index (MVI)

To quantify non-monotonic behavior in unconstrained benchmark models, the validator computes the Monotonicity Violation Index over an evenly spaced 1D partial dependence grid $x_{j,1} < x_{j, 2} < ... < x_{j, M}$:

$$\text{MVI}_{j} = \frac{\sum^{M-1}_{k=1} \text{max}\left(0, -m_{j} \cdot (\bar{f}(x_{j,k+1}) - \bar{f}(x_{j,k}))\right)}{\text{max}_{k}\bar{f}(x_{j,k}) - \text{min}_{k}\bar{f}(x_{j,k})}$$

Where $\bar{f}(x_{j,k}) = \frac{1}{N}f(x_{j,k}, \text{x}_{i, -j})$ is the marginal expectation. An $\text{MVI}_{j} > 0$ indicates local economic invalidity requiring model rejection or re-training under monotone constraints.

### 3. Purged Group Time-Series Cross-Validation

To eliminate data leakage across panel observations of middle-market obligors $i$ across years $t$, observations are grouped by obligor ID and split temporally:

$$\mathcal{D}_{\text{train}} = \{(\text{x}_{i,t}, y_{i,t}) \mid t \le T_{\text{split}} - \text{Embargo}\}, \qquad \mathcal{D}_{\text{test}} = \{(\text{x}_{i,t}, y_{i,t}) | t > T_{\text{split}}\}$$

### 4. Calibration & Hosmer-Lemeshow Goodness-of-Fit

Raw tree predictions $\hat{p}_{\text{raw}}$ are mapped to calibrated default

$\hat{p}_{\text{cal}}$ using Platt Logistic Calibration:

$$\text{logit}(\hat{p}_{\text{cal}}) = A \cdot \text{logit}(\hat{p}_{\text{raw}}) + B$$

The Hosmer-Lemeshow statistic evaluates calibration across $G$ risk deciles:

$$H = \sum^{G}_{g=1}\frac{(O_{g} - N_{g}\bar{p}_{g})^{2}}{N_{g}\bar{p}_{g}(1-\bar{p}_{g})} \sim \chi^{2}(G - 2)$$

Where $O_{g}$ is the observed defaults in decline $g$. $N_{g}$ is total obligors in decile $g$, and $\bar{p}_{g}$ is average calibrated predicted probability.

In [6]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import xgboost as xgb
import shap
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression

class XGBoostCreditModelValidator:
  """
  Independent Model Risk Validation Engine for Machine Learning (XGBoost) Credit Scoring Models.
  Evaluates Monotonicity Violations, Purged Time-Series Decay, Probability Calibration, and FCRA Adverse Action.
  """
  def __init__(self, train_df: pd.DataFrame, oot_df: pd.DataFrame, feature_cols: list, target_col: str):
    self.train_df = train_df.copy()
    self.oot_df = oot_df.copy()
    self.features = feature_cols
    self.target = target_col
    self.unconstrained_model = None
    self.constrained_model = None
    self.calibrator = None

  def fit_candidate_models(self, monotone_constraints_map: dict):
    """
    Fits both an Unconstrained XGBoost model (Front-Office style) and
    a Monotonically Constrained XGBoost model (MRM Champion candidate).
    """
    X_tr = self.train_df[self.features]
    y_tr = self.train_df[self.target]

    # 1. Unconstrained Front-Office Candidate
    self.unconstrained_model = xgb.XGBClassifier(
        n_estimators = 120, max_depth = 4, learning_rate = 0.04,
        eval_metric = "logloss", random_state = 42
    )
    self.unconstrained_model.fit(X_tr, y_tr)

    # Build constraint tuple matching feature ordering
    # e.g., tuple (+1, -1, -1, +1)
    contraint_tuple = tuple(monotone_constraints_map.get(col, 0) for col in self.features)

    # 2. Monotonically Constrained MRM Champion Model
    self.constrained_model = xgb.XGBClassifier(
        n_estimators = 120, max_depth=4, learning_rate=0.04,
        monotone_constraints = contraint_tuple,
        eval_metric="logloss", random_state=42
    )
    self.constrained_model.fit(X_tr, y_tr)

    # 3. Fit Platt Calibration Engine on Constrained Model Predictions
    raw_train_preds = self.constrained_model.predict_proba(X_tr)[:,1]
    raw_train_logits = np.log(np.clip(raw_train_preds, 1e-6, 1 - 1e-6) / (1 - np.clip(raw_train_preds, 1e-6, 1 - 1e-6)))

    self.calibrator = LogisticRegression(C=1e3)
    self.calibrator.fit(raw_train_logits.reshape(-1, 1), y_tr)

  def audit_feature_monotonicity(self, feature_name: str, expected_direction: int, grid_points: int = 50) -> dict:
    """
    Computes the Monotonicity Violation Index (MVI) via Partial Dependence Analysis.
    expected_direction: +1 for increasing PD (e.g. Leverage), -1 for decreasing PD (e.g. Coverage).
    """
    X_tr = self.train_df[self.features].copy()
    feat_min, feat_max = X_tr[feature_name].min(), X_tr[feature_name].max()
    grid = np.linspace(feat_min, feat_max, grid_points)

    unconstrained_pdp = []
    constrained_pdp = []

    for val in grid:
      X_temp = X_tr.copy()
      X_temp[feature_name] = val

      p_ununc = self.unconstrained_model.predict_proba(X_temp)[:, 1].mean()
      p_const = self.constrained_model.predict_proba(X_temp)[:, 1].mean()

      unconstrained_pdp.append(p_ununc)
      constrained_pdp.append(p_const)

    unconstrained_pdp = np.array(unconstrained_pdp)
    constrained_pdp = np.array(constrained_pdp)

    # Calculate Monotonicity Violation Index (MVI)
    unconstrained_diffs = np.diff(unconstrained_pdp)
    constrained_diffs = np.diff(constrained_pdp)

    # Violations occur when sign(diff) != expected_direction
    unconstrained_violations = np.sum(np.maximum(0, -expected_direction * unconstrained_diffs))
    unconstrained_range = np.max(unconstrained_pdp) - np.min(unconstrained_pdp) + 1e-6
    mvi_unconstrained = unconstrained_violations / unconstrained_range

    constrained_violations = np.sum(np.maximum(0, -expected_direction * constrained_diffs))
    constrained_range = np.max(constrained_pdp) - np.min(constrained_pdp) + 1e-6
    mvi_constrained = constrained_violations / constrained_range

    return {
        "Audited_Feature": feature_name,
        "Expected_Direction": "Increasing (+1)" if expected_direction == 1 else "Decreasing (-1)",
        "Unconstrained_MVI": round(float(mvi_unconstrained), 5),
        "Constrained_MVI": round(float(mvi_constrained), 5),
        "Unconstrained_Pass": mvi_unconstrained < 1e-4,
        "Constrained_Pass": mvi_constrained < 1e-4
    }

  def predict_calibrated_pd(self, model, X_data: pd.DataFrame) -> np.ndarray:
    """Helper to output calibrated probability of default."""
    raw_preds = model.predict_proba(X_data)[:, 1]
    raw_logits = np.log(np.clip(raw_preds, 1e-6, 1- 1e-6) / (1 - np.clip(raw_preds, 1e-6, 1 - 1e-6)))
    return self.calibrator.predict_proba(raw_logits.reshape(-1, 1))[:, 1]

  def evaluate_hosmer_lemeshow_calibration(self, y_true: np.ndarray, y_prob: np.ndarray, num_groups: int = 10) -> dict:
    """
    Evaluates Hosmer-Lemeshow Goodness-of-Fit calibration test.
    """
    df_cal = pd.DataFrame({"y_true": y_true, "y_prob": y_prob})
    df_cal['decile'] = pd.qcut(df_cal['y_prob'], q=num_groups, duplicates='drop')

    hl_stat = 0.0
    for name, group in df_cal.groupby('decile'):
      n_g = len(group)
      o_g = group['y_true'].sum()
      p_g = group['y_prob'].mean()

      num = (o_g - n_g * p_g) ** 2
      den = n_g * p_g * (1.0 - p_g) + 1e-6
      hl_stat += num / den

    p_value = 1.0 - stats.chi2.cdf(hl_stat, df=num_groups - 2)

    return {
        "HL_Statistic": round(float(hl_stat), 4),
        "P_Value": round(float(p_value), 5),
        "Calibration_Status": "WELL_CALIBRATED_PASS" if p_value > 0.05 else "MISCALIBRATED_FAIL"
    }

  def generate_fcra_adverse_action(self, applicant_df: pd.DataFrame, top_reasons: int = 3) -> list:
    """
    Generates FCRA/ECOA compliant principal denial reason codes using SHAP tree explainer.
    """
    explainer = shap.TreeExplainer(self.constrained_model)
    shap_values = explainer.shap_values(applicant_df[self.features])

    reasons_list = []
    for i in range(len(applicant_df)):
      row_shap = shap_values[i]
      row_vals = applicant_df[self.features].iloc[i].values

      # Map features with positive SHAP (pushed PD HIGHER)
      feature_impacts = list(zip(self.features, row_shap, row_vals))
      adverse_drivers = sorted(feature_impacts, key=lambda x: x[1], reverse=True)

      applicant_reasons = []
      for feat, s_val, val in adverse_drivers[:top_reasons]:
        applicant_reasons.append({
            "Feature": feat,
            "Observed_Value": round(float(val), 3),
            "SHAP_PD_Increase_Contribution": round(float(s_val), 4)
        })
      reasons_list.append(applicant_reasons)

    return reasons_list

# --- Example Production Demonstration ---
if __name__ == "__main__":
  np.random.seed(42)
  n_train = 6000
  n_oot = 2000

  # 1. Simulate Unrated Middle-Market Borrow Dataset
  # Features: Leverage (Total Debt / EBITDA), Interest Coverage Ratio (DSCR), EBITDA Margin, Liquidity Ratio
  def generate_middle_market_data(n_samples, macro_stress=0.0):
    leverage = np.random.gamma(shape=3.5, scale=1.2, size=n_samples) + macro_stress
    icr = np.random.exponential(scale=2.5, size=n_samples) + 0.5 - (0.3 * macro_stress)
    icr = np.clip(icr, 0.2, 12.0)
    ebitda_margin = np.random.normal(0.18, 0.08, n_samples)
    liquidity_ratio = np.random.uniform(0.05, 0.40, n_samples)

    # Underlying Non-Linear Data Generating Process for Default
    logit_p = (
        -3.8
        + 0.55 * leverage
        - 0.85 * icr
        + 0.12 * (leverage / (icr + 0.1)) # Non-linear interaction
        - 2.5 * ebitda_margin
        - 1.8 * liquidity_ratio
    )
    prob_default = 1.0 / (1.0 + np.exp(-logit_p))
    defaults = np.random.binomial(1, prob_default)

    return pd.DataFrame({
        "Total_Debt_EBITDA": leverage,
        "Interest_Coverage_Ratio": icr,
        "EBITDA_Margin": ebitda_margin,
        "Liquidity_Ratio": liquidity_ratio,
        "Default_Flag": defaults
    })

  dev_data = generate_middle_market_data(n_train, macro_stress=0.0)
  oot_data = generate_middle_market_data(n_oot, macro_stress=0.8) # Out-of_Time Macro Stress Shift

  feature_cols = ["Total_Debt_EBITDA", "Interest_Coverage_Ratio", "EBITDA_Margin", "Liquidity_Ratio"]

  # Define Economic Monotonicity Expectations
  # Total_Debt_EBITDA: +1 (Increasing Leverage -> Higher PD)
  # Interest_Coverage_Ratio: -1 (Higher Coverage -> Lower PD)
  # EBITDA_Margin: -1 (Higher Profitability -> Lower PD)
  # Liquidity_Ratio: -1 (Higher Liquidity -> Lower PD)
  monotonicity_map = {
      "Total_Debt_EBITDA": 1,
      "Interest_Coverage_Ratio": -1,
      "EBITDA_Margin": -1,
      "Liquidity_Ratio": -1
  }

  # Initialize Engine
  validator = XGBoostCreditModelValidator(dev_data, oot_data, feature_cols, target_col="Default_Flag")
  validator.fit_candidate_models(monotonicity_map)

  # Step 1: Feature Monotonicity Audit (MVI Test)
  print("=== Step 1: Feature Monotonicity Audit (PDP MVI Test) ===")
  mvi_results = []
  for feat, direction in monotonicity_map.items():
    res = validator.audit_feature_monotonicity(feat, direction)
    mvi_results.append(res)
  print(pd.DataFrame(mvi_results).to_string(index=False))

  # Step 2: Performance & Gini Decay Audit (In-Sample vs Out-of-Time)
  print("=== Step 2: Discriminatory Power & Performance Decay Audit ===")
  y_tr_true = dev_data["Default_Flag"].values
  y_oot_true = oot_data["Default_Flag"].values

  # Predictions
  p_tr_unconstrained = validator.unconstrained_model.predict_proba(dev_data[feature_cols])[:, 1]
  p_oot_unconstrained = validator.unconstrained_model.predict_proba(oot_data[feature_cols])[:, 1]

  p_tr_constrained = validator.predict_calibrated_pd(validator.constrained_model, dev_data[feature_cols])
  p_oot_constrained = validator.predict_calibrated_pd(validator.constrained_model, oot_data[feature_cols])

  gini_tr_unconstrained = 2 * roc_auc_score(y_tr_true, p_tr_unconstrained) - 1
  gini_oot_unconstrained = 2 * roc_auc_score(y_oot_true, p_oot_unconstrained) - 1

  gini_tr_constrained = 2 * roc_auc_score(y_tr_true, p_tr_constrained) - 1
  gini_oot_constrained = 2 * roc_auc_score(y_oot_true, p_oot_constrained) - 1

  perf_df = pd.DataFrame([
      {
          "Model_Type": "Unconstrained Front-Office Candidate",
          "In_Sample_Gini": round(gini_tr_unconstrained, 4),
          "OOT_Gini": round(gini_oot_unconstrained, 4),
          "Gini_Relative_Decay": f"{((gini_tr_unconstrained - gini_oot_unconstrained)/gini_tr_unconstrained)*100:.2f}%",
          "OOT_Brier_Score": round(brier_score_loss(y_oot_true, p_oot_unconstrained), 5)
      },
      {    "Model_Type": "Monotonic & Calibrated MRM Champion",
          "In_Sample_Gini": round(gini_tr_constrained, 4),
          "OOT_Gini": round(gini_oot_constrained, 4),
          "Gini_Relative_Decay": f"{((gini_tr_constrained - gini_oot_constrained)/gini_tr_constrained)*100:.2f}%",
          "OOT_Brier_Score": round(brier_score_loss(y_oot_true, p_oot_constrained), 5)
      }
      ])
  print(perf_df.to_string(index=False))

  # Step 3: Probability Calibration Test (Hosmer-Lemeshow)
  print("\n=== Step 3: Probability Calibration Audit (Hosmer-Lemeshow Test) ===")
  hl_unconstrained = validator.evaluate_hosmer_lemeshow_calibration(y_oot_true, p_oot_unconstrained)
  hl_constrained = validator.evaluate_hosmer_lemeshow_calibration(y_oot_true, p_oot_constrained)

  print(f"Unconstrained Model HL Test: Stat = {hl_unconstrained['HL_Statistic']} | P-Value = {hl_unconstrained['P_Value']} -> Status: {hl_unconstrained['Calibration_Status']}")
  print(f"Calibrated Champion HL Test: Stat = {hl_constrained['HL_Statistic']} | P-Value = {hl_constrained['P_Value']} -> Status: {hl_constrained['Calibration_Status']}")

  # Step 4: FCRA Adverse Action Reason Code Extraction
  print("\n=== Step 4: FCRA Adverse Action Reason Codes (Sample Declined Borrower) ===")
  sample_declined_borrower = oot_data.iloc[[5]]
  reasons = validator.generate_fcra_adverse_action(sample_declined_borrower, top_reasons=3)

  print(f"Sample Applicant Metrics:")
  print(sample_declined_borrower[feature_cols].to_string(index=False))
  print("\n Top FCRA Denial Reasons (SHAP Contributions):")
  for r in reasons[0]:
    print(f" - Principal Factor: {r['Feature']} (Value: {r['Observed_Value']}) -> Added +{r['SHAP_PD_Increase_Contribution']*100:.2f}% to Log-Odds PD")




=== Step 1: Feature Monotonicity Audit (PDP MVI Test) ===
        Audited_Feature Expected_Direction  Unconstrained_MVI  Constrained_MVI  Unconstrained_Pass  Constrained_Pass
      Total_Debt_EBITDA    Increasing (+1)            0.03795              0.0               False              True
Interest_Coverage_Ratio    Decreasing (-1)            0.02594              0.0               False              True
          EBITDA_Margin    Decreasing (-1)            0.19232              0.0               False              True
        Liquidity_Ratio    Decreasing (-1)            0.42405              0.0               False              True
=== Step 2: Discriminatory Power & Performance Decay Audit ===
                          Model_Type  In_Sample_Gini  OOT_Gini Gini_Relative_Decay  OOT_Brier_Score
Unconstrained Front-Office Candidate          0.9115    0.7824              14.16%          0.05934
 Monotonic & Calibrated MRM Champion          0.8731    0.7919               9.30%          0.

/tmp/ipykernel_625/4253974971.py:118: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df_cal.groupby('decile'):
/tmp/ipykernel_625/4253974971.py:118: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df_cal.groupby('decile'):


# Model Interpretation

### Step 1: Feature Montonicity Audit (PDP MVI Test)

* **Unconstrained Model Failure**: The unconstrained candidate violates fundamental financial logic across all four key risk drivers, yielding non-zero Mean Variation Index (MVI) values. The highest violations occur in Liquidity_Ratio (MVI = 0.42405) and EBITDA_Margin (MVI = 0.19232). In practice, this creates non-monotonic partial dependence curves where marginal improvements in liquidity or margin paradoxically increase the predicted probability of default (PD) within certain localized decision boundaries.

* **Constrained Champion Enforcement**: The application of hard monotonic constraints successfully forces strict economic alignment across all features (MVI = 0.0000 across all variables). This ensures that increases in risk-mitigating metrics (Interest_Coverage_Ratio, EBITDA_Margin, Liquidity_Ratio) strictly decreases predicted PD, while increases in Total_Debt_EBITA strictly increases PD.

### Step 2: Discriminatory Power & Performance Decay Audit

* **Overfitting in Unconstrained Model**: The unconstrained model exhibits significant in-sample overfitting, achieving an In-Sample Gini of 0.9115 but dropping to an Out-of-Time (OOT) Gini of 0.7824, a performance decay of 14.16%.

* **Generalization Advantage of Monotonic Constraints**: Enforcing monotonic constraints acts as a stuctural regularizer. While the constrained champion exhibits lower in-sample discrimination (In-Sample Gini = 0.8731), it generalizes substantially better to unseen data, achieving a higher OOT Gini (0.7919) and a lower decay rate (9.30%).

* **Overall Loss**: Both models demonstrate nearly identical OOT Brier Scores (~0.05935), indicating overall mean squared error accuracy is preserved despite the structural constraints.

### Step 3: Probability Calibration Audit (Hosmer-Lemeshow Test)

* **Critical Governance Deficiency**: Both models fail the Hosmer-Lemeshow (HL) calibration test at the 5% significance level (p < 0.05).
  * Unconstrained Model: HL Stat = 16.9412 (p = 0.03073)
  * Calibrated Champion: HL Stat = 25.8466 (p = 0.00112)

* **Model Risk Management (MRM) Concern**: The calibrated champion actually demonstrates worse goodness-of-fit across risk deciles than the unconstrained model, despite calibration post-processing. A p-value of 0.00112 confirms systematic divergence between predicted PDs and actual observed default rates across risk buckets. This model cannot be approved for production pricing, provisioning, or capital allocation in its current state without re-calibrating via isotonic regression or Platt scaling on a non-overfitted validation set.

### Step 4: FCRA Adverse Action Reason Codes (Explainability Audit)
* **Local Attribution Analysis**: Using SHAP (Shapley Additive exPlanations) values on the sample declined borrower (Total_Debt_EBITDA = 4.87, Interest_Coverage_Ratio = 1.10, EBITDA_Margin = 0.225, Liquidity_Ratio = 0.105):
  1. **Primary Denial Factor**: Weak Interest_Coverage_Ratio (1.10) is the dominant contributor to default risk, adding +56.85% to the log-odds of default.
  2. **Secondary Denial Factor**: Low Liquidity_Ratio (0.105) adds +17.50% to the log-odds of default.
  3. **Mitigating Factor**: EBITDA_Margin (0.225) serves as a partial buffer, reducing log-odds default risk by -25.58%.
* **Regulatory Compliance**: The SHAP-based decomposition provides a mathematically defensible, rank-ordered mechanism for generating Fair Credit Reporting Act (FCRA) adverse action notices based on true marginal contribution to risk.